# Creating Test Dataset

Purpose

* Enable fast end-to-end pipeline runs
* Provide a minimal dataset that fits within GitHub size limits

Data provenance

This test dataset is derived from publicly available reference resources:
* Human genome assembly hg38
* Mouse genome assembly mm39
* Genome alignment chains: hg38 ↔ mm39
* GENCODE transcript annotations (BED12)

See repository README for exact download sources and version numbers.

It requires TOGA results from the full run.
To generate them pls call

```bash
./curia.py --ref-bed12 input_data/reference_annotation/hg38.input.w.tRNA.bed \
--biomart-tsv input_data/reference_annotation/hg38.transcript_metadata.tsv \
--chain input_data/chains/hg38.mm39.allfilled.chain.gz \
--ref-2bit input_data/2bit/hg38.2bit \
--query-2bit input_data/2bit/mm39.2bit  \
--gpu-max-batch 128 --short-max-workers 100 \
--gpu-logger \
--output-dir input_data/supply \
--stop-after-toga
```

In [1]:
from dataclasses import replace
from pathlib import Path
import pyrion
import pandas as pd

from pyrion import GenomeAlignmentsCollection, TwoBitAccessor
from pyrion.core import TranscriptsCollection

In [2]:
# Paths to original data files
ref_bed12 = Path("./reference_annotation/hg38.input.w.tRNA.bed")
biomart_tsv = Path("./reference_annotation/hg38.transcript_metadata.tsv")
biomart_tsv_filtered = Path("./reference_annotation/hg38.transcript_metadata.filt.tsv")
chain_file = Path("./chains/hg38.mm39.allfilled.chain.gz")
ref_2bit = Path("./2bit/hg38.2bit")
query_2bit = Path("./2bit/mm39.2bit")
orthologous_regions = Path("./supply/toga_mini_results/toga_orthologous_regions.tsv")

# Chromosomes to extract
hg38_chroms = ["chr11", "chrX"]
mm39_chroms = ["chr19", "chr7", "chrX"]

In [18]:
genes_to_preserve = {
    "ENSG00000251562": "MALAT1",
    "ENSG00000245532": "NEAT1",
    "ENSG00000229807": "XIST",
    "ENSG00000130600": "H19",
    "ENSG00000241743": "XACT",
    "ENSG00000269821": "KCNQ1OT1"
}

In [3]:
# output files paths
test_chain_file = "./chains/test_sample.chain"
test_bed_file = "./reference_annotation/test_sample.bed"
test_metadata = "./reference_annotation/test_sample.metadata.tsv"
test_hg38_2bit = "./2bit/hg38.test.subset.2bit"
test_mm39_2bit = "./2bit/mm39.test.subset.2bit"

## Filter chain file

In [4]:
# parsed_chain = pyrion.io.read_chain_file(chain_file)
# print(parsed_chain)

In [5]:
# # pyrion DONE: add better API to pyrion to handle such scenarios, now apply
# TODO: potentially a chain saving bug
# filtered_chains = []
# for c in hg38_chroms:
#     chains = [x for x in parsed_chain.get_by_target_chrom(c) if x.q_chrom in mm39_chroms]
#     filtered_chains.extend(chains)
# filtered_chains_collection = GenomeAlignmentsCollection(filtered_chains)
# print(filtered_chains_collection)
# filtered_chains_collection.save_to_chain(test_chain_file)

In [6]:
!chainFilter chains/hg38.mm39.allfilled.chain.gz -t=chr11,chrX -q=chr19,chr7,chrX -minScore=15000 > chains/test_sample.chain

In [7]:
!gzip ./chains/test_sample.chain

./chains/test_sample.chain.gz already exists -- do you wish to overwrite (y or n)? ^C


# Filter bed file

In [8]:
# for simplicity -> get rid of .X version numbers everywhere
from pyrion import read_gene_data

def id_mapping(id_str: str) -> str:
    return id_str.split(".")[0]

transcripts_without_version = [
    replace(t, id=id_mapping(t.id)) for t in pyrion.io.read_bed12_file(ref_bed12).transcripts
]
bed12_data = TranscriptsCollection(transcripts=transcripts_without_version)
print(f"Total transcripts in BED12: {len(bed12_data)}")

metadata = pd.read_csv(biomart_tsv, sep='\t')
metadata[['transcript_id', 'gene_id']] = metadata[['transcript_id', 'gene_id']].apply(
    lambda col: col.str.replace(r'\.\d+$', '', regex=True))
metadata.to_csv(biomart_tsv_filtered, sep='\t', index=False)

gene_data = read_gene_data(
    biomart_tsv_filtered,
    transcript_id_column="transcript_id",
    gene_column="gene_id",
    transcript_type_column="transcript_biotype",
    separator="\t",
)

Total transcripts in BED12: 386300


In [9]:
# pyrion DONE/ APPLY: better filters for chroms (like chroms list)
transcripts_on_target_chroms = [t for t in bed12_data.transcripts if t.chrom in hg38_chroms]
transcripts_collection = TranscriptsCollection(transcripts_on_target_chroms)
transcripts_collection.bind_gene_data(gene_data)
print(transcripts_collection)

TranscriptsCollection: 31,450 transcripts across 2 chromosomes, 31,450 coding (100.0%)


In [10]:
# pyrion DONE: APPLY biotype and others filters for Tcollections
protein_coding_transcripts = TranscriptsCollection([t for t in transcripts_collection if t.biotype == "protein_coding"])
lncrna_transcripts = TranscriptsCollection([t for t in transcripts_collection if t.biotype == "lncRNA"])
snorna_transcripts = TranscriptsCollection([t for t in transcripts_collection if t.biotype == "snoRNA"])
trna_transcripts = TranscriptsCollection([t for t in transcripts_collection if t.biotype == "tRNA"])
mirna_transcripts = TranscriptsCollection([t for t in transcripts_collection if t.biotype == "miRNA"])

print(f"Len protein coding transcripts: {len(protein_coding_transcripts)}")
print(f"Len lncRNA transcripts: {len(lncrna_transcripts)}")
print(f"Len snoRNA transcripts: {len(snorna_transcripts)}")
print(f"Len tRNA transcripts: {len(protein_coding_transcripts)}")
print(f"Len miRNA transcripts: {len(protein_coding_transcripts)}")

protein_coding_transcripts.bind_gene_data(gene_data)
lncrna_transcripts.bind_gene_data(gene_data)
snorna_transcripts.bind_gene_data(gene_data)
trna_transcripts.bind_gene_data(gene_data)
mirna_transcripts.bind_gene_data(gene_data)

Len protein coding transcripts: 9363
Len lncRNA transcripts: 11881
Len snoRNA transcripts: 98
Len tRNA transcripts: 9363
Len miRNA transcripts: 9363


In [ ]:
# Load orthologous regions to understand which transcripts map where
# Filter transcripts that have orths on the selected mm39 chromosomes
orth_regions = pd.read_csv(orthologous_regions, sep='\t')
orth_regions['transcript_id'] = orth_regions['transcript_id'].str.replace(r'^U_', '', regex=True).str.replace(r'\.\d+$', '', regex=True)
orth_regions = orth_regions.rename(columns={'transcript_id': 'gene_id'})

print(f"Total orthologous regions: {len(orth_regions)}")
orth_regions['chrom'] = orth_regions['region'].str.split(':').str[0]
orth_regions_filtered = orth_regions[orth_regions['chrom'].isin(mm39_chroms)]

In [12]:

print(f"Filtered orthologous regions: {len(orth_regions_filtered)}")
filtered_genes = set(orth_regions_filtered.gene_id)
print(len(filtered_genes))

Total orthologous regions: 90246
Filtered orthologous regions: 12942
8119


In [13]:
# out of 6328 genes mapping to the mouse chromosomes of interest, let's find those from target regions that could be interesting
protein_coding_genes_mapped_to_mm39 = [x for x in [protein_coding_transcripts.get_gene_by_id(g) for g in filtered_genes] if x]
print(f"Protein coding genes mapping from hg38 to m39 (selected chromosomes only): {len(protein_coding_genes_mapped_to_mm39)}")

lncrna_genes_mapped_to_mm39 = [x for x in [lncrna_transcripts.get_gene_by_id(g) for g in filtered_genes] if x]
print(f"lncRNA genes mapping from hg38 to m39 (selected chromosomes only): {len(lncrna_genes_mapped_to_mm39)}")

trna_genes_mapped_to_mm39 = [x for x in [trna_transcripts.get_gene_by_id(g) for g in filtered_genes] if x]
print(f"tRNA genes mapping from hg38 to m39 (selected chromosomes only): {len(trna_genes_mapped_to_mm39)}")

snorna_genes_mapped_to_mm39 = [x for x in [snorna_transcripts.get_gene_by_id(g) for g in filtered_genes] if x]
print(f"snoRNA genes mapping from hg38 to m39 (selected chromosomes only): {len(snorna_genes_mapped_to_mm39)}")

mirna_genes_mapped_to_mm39 = [x for x in [mirna_transcripts.get_gene_by_id(g) for g in filtered_genes] if x]
print(f"miRNA genes mapping from hg38 to m39 (selected chromosomes only): {len(mirna_genes_mapped_to_mm39)}")

Protein coding genes mapping from hg38 to m39 (selected chromosomes only): 1461
lncRNA genes mapping from hg38 to m39 (selected chromosomes only): 1315
tRNA genes mapping from hg38 to m39 (selected chromosomes only): 13
snoRNA genes mapping from hg38 to m39 (selected chromosomes only): 39
miRNA genes mapping from hg38 to m39 (selected chromosomes only): 126


In [29]:
for g, name in genes_to_preserve.items():
    print(g)
    print(lncrna_transcripts.get_gene_by_id(g))
    print(g in filtered_genes)

ENSG00000251562
Gene(id=ENSG00000251562, chr11:65497605-65508073:1, 73 transcripts)
True
ENSG00000245532
Gene(id=ENSG00000245532, chr11:65422773-65445540:1, 16 transcripts)
True
ENSG00000229807
Gene(id=ENSG00000229807, chrX:73820648-73852714:-1, 33 transcripts)
True
ENSG00000130600
Gene(id=ENSG00000130600, chr11:1995164-2004552:-1, 39 transcripts)
True
ENSG00000241743
Gene(id=ENSG00000241743, chrX:113616299-114059289:-1, 13 transcripts)
True
ENSG00000269821
Gene(id=ENSG00000269821, chr11:2597307-2700003:-1, 14 transcripts)
True


In [39]:
lncrna_genes_to_save = []  # keep max 100
counter_added = 0
genes_to_preserve_set = set(genes_to_preserve.keys())
for g in lncrna_genes_mapped_to_mm39:
    # print(g.gene_id)
    if g.gene_id in genes_to_preserve_set:
        print(g, genes_to_preserve[g.gene_id])
        lncrna_genes_to_save.append(g)
    counter_added += 1
    if counter_added <= 100:
        lncrna_genes_to_save.append(g)

Gene(id=ENSG00000251562, chr11:65497605-65508073:1, 73 transcripts) MALAT1
Gene(id=ENSG00000269821, chr11:2597307-2700003:-1, 14 transcripts) KCNQ1OT1
Gene(id=ENSG00000241743, chrX:113616299-114059289:-1, 13 transcripts) XACT
Gene(id=ENSG00000130600, chr11:1995164-2004552:-1, 39 transcripts) H19
Gene(id=ENSG00000245532, chr11:65422773-65445540:1, 16 transcripts) NEAT1
Gene(id=ENSG00000229807, chrX:73820648-73852714:-1, 33 transcripts) XIST


In [40]:
from pyrion.core import longest_cds_canonizer

# DONE: pyrion: APPLY append and extend methods for collections
selected_transcripts_list = []
mapped_genes_sets = [protein_coding_genes_mapped_to_mm39, lncrna_genes_to_save, trna_genes_mapped_to_mm39, snorna_genes_mapped_to_mm39, mirna_genes_mapped_to_mm39]
for gene_set in mapped_genes_sets:
    for gene in gene_set:
        gene.apply_canonizer(longest_cds_canonizer)
        selected_transcripts_list.append(gene.canonical_transcript)

selected_transcripts = TranscriptsCollection(selected_transcripts_list)
selected_transcripts.save_to_bed12(test_bed_file)
print(f"Saved {len(selected_transcripts)} transcripts to BED12 file")

Saved 1745 transcripts to BED12 file


In [41]:
# save transcript metadata
# pyrion DONE:  apply metadata ops: save trimmed metadata for this exact collection
saved_transcript_ids = set(t.id for t in selected_transcripts.transcripts)
gene_data._transcript_to_gene = {k: v for k, v in gene_data._transcript_to_gene.items() if k in saved_transcript_ids}
gene_data._transcript_to_biotype = {k: v for k, v in gene_data._transcript_to_biotype.items() if k in saved_transcript_ids}

selected_transcripts.bind_gene_data(gene_data)
selected_transcripts.save_biodata(test_metadata)

# Filter 2bit files

In [ ]:
# ADDED: pyrion, add a way to fetch the whole chrom from 2bit
# hg38_chroms = ["chr11", "chrX"]
# mm39_chroms = ["chr19", "chr7", "chrX"]

In [ ]:
%%bash
mkdir temp
twoBitToFa 2bit/hg38.2bit -seq=chr11 temp/hg38.chr11.fa
twoBitToFa 2bit/hg38.2bit -seq=chrX temp/hg38.chrX.fa
cat temp/hg38.chr11.fa temp/hg38.chrX.fa > temp/hg38.test.subset.fa
faToTwoBit temp/hg38.test.subset.fa 2bit/hg38.test.subset.2bit
rm temp/hg38.*

twoBitToFa 2bit/mm39.2bit -seq=chr19 temp/mm39.chr19.fa
twoBitToFa 2bit/mm39.2bit -seq=chr7 temp/mm39.chr7.fa
twoBitToFa 2bit/mm39.2bit -seq=chrX temp/mm39.chrX.fa
cat temp/mm39.chr19.fa temp/mm39.chr7.fa  temp/mm39.chrX.fa > temp/mm39.test.subset.fa
faToTwoBit temp/mm39.test.subset.fa 2bit/mm39.test.subset.2bit

rm -rf temp

In [42]:
from pyrion.io.twobit import TwoBitAccessor

test_hg38_2bit_accessor = TwoBitAccessor(test_hg38_2bit)
print(test_hg38_2bit_accessor)

test_mm39_2bit_accessor = TwoBitAccessor(test_mm39_2bit)
print(test_mm39_2bit_accessor)

TwoBitAccessor('./2bit/hg38.test.subset.2bit', 2 chromosomes [chr11, chrX], 291.1Mb)
TwoBitAccessor('./2bit/mm39.test.subset.2bit', 3 chromosomes [chr19, chr7, chrX], 375.9Mb)


In [44]:
!rm reference_annotation/hg38.transcript_metadata.filt.tsv
!rm chains/test_sample.chain

rm: reference_annotation/hg38.transcript_metadata.filt.tsv: No such file or directory


# Command to execute

From the project root:

```bash
./curia.py --ref-bed12 input_data/reference_annotation/test_sample.bed \
--biomart-tsv input_data/reference_annotation/test_sample.metadata.tsv \
--chain input_data/chains/test_sample.chain.gz \
--ref-2bit input_data/2bit/hg38.test.subset.2bit \
--query-2bit input_data/2bit/mm39.test.subset.2bit \
--gpu-max-batch 128 --short-max-workers 100 \
--gpu-logger \
--output-dir quick_test
```

# Full run on full data


```bash
./curia.py --ref-bed12 input_data/reference_annotation/hg38.input.w.tRNA.bed \
--biomart-tsv input_data/reference_annotation/hg38.transcript_metadata.tsv \
--chain input_data/chains/hg38.mm39.allfilled.chain.gz \
--ref-2bit input_data/2bit/hg38.2bit \
--query-2bit input_data/2bit/mm39.2bit  \
--gpu-max-batch 128 --short-max-workers 100 \
--gpu-logger \
--output-dir big_test
```